# 02 - Isolating Variation

In [ ]:
%autosave 0

In [ ]:
import os
import sys
from pathlib import Path

# Set the base project directory
base_proj_dir = Path("/projects/site/pred/ihb-g-deco/USERS/adaml9/intestine_fate/notebooks/tf_ko_panel_analysis")

# Append necessary paths for module imports
sys.path.append("/projects/site/pred/ihb-g-deco/USERS/adaml9/bioinfo-blueprint/src/python")
sys.path.append(str(base_proj_dir))

import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import snapseed as snap
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import marsilea as ma
import marsilea.plotter as mp
import cellmapper
import plotting.settings
from dynaconf import Dynaconf

# Load settings
settings = Dynaconf(
    settings_files=[base_proj_dir / 'settings.toml', base_proj_dir / '.secrets.toml'],
)

sc.settings.figdir = settings.ANALYSIS.figures_dir

In [ ]:
data_dir = Path(settings.IO.combined_data) / "all-sample" / "DGE_filtered"
anndata_dir = Path(settings.IO.anndata)

## Load data 

In [ ]:
adata = sc.read(anndata_dir / "tf_ko_panel_contrastiveVI_raw_cystic_ann.h5ad")

In [ ]:
adata_query = adata[~adata.obs["condition"].isin(["Control"])]
adata_ref = adata[adata.obs["condition"].isin(["Control"])]

In [ ]:
adata_query.shape, adata_ref.shape

## Annotate using controls

In [ ]:
adata_controls = sc.read(anndata_dir / "tf_ko_panel_combined_controls_annotated.h5ad")

In [ ]:
adata_ref.obs["annotation"] = adata_controls[adata_ref.obs.index].obs["cell_type"].values

In [ ]:
sc.pl.embedding(adata_ref, basis="X_umap_shared_rep", color="annotation", legend_loc="on data")

In [ ]:
cmap = cellmapper.CellMapper(adata_query, adata_ref).map(
    obs_keys=["annotation", ], use_rep="shared_rep", knn_method="pynndescent"
)
cmap

In [ ]:
sc.pl.embedding(adata_query, basis="X_umap_shared_rep", color=["annotation_pred", "annotation_conf"], legend_loc="on data")

In [ ]:
# Save the annotated query adata
adata_query.write(anndata_dir / "tf_ko_panel_contrastiveVI_perturbed_annotated.h5ad")

# Save the annotated reference adata
adata_ref.write(anndata_dir / "tf_ko_panel_contrastiveVI_controls_annotated.h5ad")

In [ ]:
# Add all annotations to the original adata
adata.obs.loc[adata_query.obs.index, "annotation"] = adata_query.obs["annotation_pred"]
adata.obs.loc[adata_query.obs.index, "annotation_conf"] = adata_query.obs["annotation_conf"]
adata.obs.loc[adata_ref.obs.index, "annotation"] = adata_ref.obs["annotation"]
adata.obs.loc[adata_ref.obs.index, "annotation_conf"] = 1.0

In [ ]:
# Convert cystic 0/1 to string labels
adata.obs["cystic"] = adata.obs["cystic"].map({0: "budding", 1: "cystic"})
# Combine cystic/budding annotation with annotation
adata.obs["cell_type"] = adata.obs["annotation"].astype(str) + " - " + adata.obs["cystic"].astype(str)

In [ ]:
# Save the annotated adata
adata.write(anndata_dir / "tf_ko_panel_contrastiveVI_raw_annotated.h5ad")

## Visualize jointly

In [ ]:
adata = sc.read(anndata_dir / "tf_ko_panel_contrastiveVI_raw_annotated.h5ad")

In [ ]:
adata

In [ ]:
adata.obs["condition"].unique()

In [ ]:
list(adata.obs["annotation"].unique())

In [ ]:
adata.obs["is_eec"] = adata.obs["annotation"].isin(["Early EEC Progenitors", "Late EEC Progenitors", "EC Cells", "X Cells", "D Cells", "I/N Cells", "K Cells"])
adata.obs["is_secretory"] = adata.obs["annotation"].isin(["Early EEC Progenitors", "Late EEC Progenitors", "EC Cells", "X Cells", "D Cells", "I/N Cells", "K Cells", "Goblet Cells", "Paneth Cells"])
adata.obs["is_control"] = adata.obs["condition"].apply(lambda x: "Control" in x)

In [ ]:
adata.obs["is_eec"].value_counts(), adata.obs["is_secretory"].value_counts(), adata.obs["is_eec"].value_counts(normalize=True), adata.obs["is_secretory"].value_counts(normalize=True)

In [ ]:
import matplotlib.colors as mcolors
import colorsys

def lighten(color, amount=0.5):
    """
    Lighten color by blending with white.
    amount=0 → no change, amount=1 → white
    """
    c = mcolors.to_rgb(color)
    return tuple((1 - amount) * x + amount * 1 for x in c)

def darken(color, amount=0.5):
    """
    Darken color by blending with black.
    amount=0 → no change, amount=1 → black
    """
    c = mcolors.to_rgb(color)
    return tuple((1 - amount) * x for x in c)

base_ct_colors = {}
for section, values in settings["ct_palette"].items():
    base_ct_colors.update(values)
    
ct_palette = {}

for ct, base in base_ct_colors.items():
    # convert base to RGB
    base_rgb = mcolors.to_rgb(base)

    ct_palette[f"{ct} - cystic"] = lighten(base, 0.4)  # lighter
    ct_palette[f"{ct} - budding"] = darken(base, 0.3)  # darker

In [ ]:
sc.pl.embedding(adata, 
                basis="X_umap_shared_rep", 
                color="cell_type", 
                palette=ct_palette, 
                frameon=False,
                show=True,
                save="_shared_rep_label_transfer_annotation.pdf")

In [ ]:
adata

In [ ]:
adata.obs["is_control"] = adata.obs["is_control"].astype(str)
adata.obs["is_control"] = pd.Categorical(values=adata.obs["is_control"], categories=["False", "True"], ordered=True)
adata = adata[adata.obs.sort_values("is_control").index]

In [ ]:
sc.pl.embedding(adata, 
                basis="X_umap_shared_rep", 
                color="is_control", 
                palette={"False": "lightgrey", "True": "black"},
                frameon=False,
                groups=["False", "True"],
                show=True,
                size=1,
                save="_shared_rep_is_control.pdf")

In [ ]:
sc.pl.embedding(adata, 
                basis="X_umap_shared_rep", 
                color="cell_type", 
                palette=ct_palette, 
                frameon=False,
                show=True,
                save="_shared_rep_label_transfer_annotation.pdf")

In [ ]:
adata.obs["condition"].unique()

In [ ]:
sc.pl.embedding(adata, 
                basis="X_umap", 
                color="condition", 
                frameon=False,
                show=True,
                save="_orig_rep_condition.pdf")

In [ ]:
cond_colors = {
    #"Atoh1": "#1e4b8f",  
    #"Lmx1b": "#d0933d",  
    #"Rfx6": "#33597d",
    #"Znf800": "#3e713c",
    "Percc1": "#8d263f"
}

In [ ]:
for cond_target in cond_colors.keys():
    adata_subset = adata[adata.obs["condition"].isin(["Control", cond_target])]
    adata_subset.obs["condition"] = pd.Categorical(values=adata_subset.obs["condition"], categories=["Control", cond_target], ordered=True)
    adata_subset = adata_subset[adata_subset.obs.sort_values("condition").index]
    
    sc.pl.embedding(adata_subset, 
                    basis="X_umap_shared_rep", 
                    color="condition", 
                    palette={"Control": "lightgrey", cond_target: cond_colors[cond_target]},
                    frameon=False,
                    groups=["Control", cond_target],
                    show=True,
                    size=5,
                    save=f"_shared_rep_{cond_target}_vs_control.pdf")

In [ ]:
sc.pl.embedding(adata, 
                basis="X_umap_shared_rep", 
                color="annotation_conf", 
                frameon=False,
                cmap="RdPu",
                show=True,
                save="_shared_rep_label_transfer_annotation_conf.pdf")

In [ ]:
sc.pl.embedding(adata, 
                basis="X_umap_shared_rep", 
                color="cystic_score", 
                frameon=False,
                cmap="Greys",
                show=True,
                sort_order=True,
                size=1,
                save="_shared_rep_label_transfer_cystic_score.pdf")

In [ ]:
adata

In [ ]:
sc.pl.embedding(adata, 
                basis="X_umap", 
                color="cell_type", 
                palette=ct_palette, 
                frameon=False,
                show=True,
                save="_orig_rep_label_transfer_annotation.pdf")

## Subset to EECs

In [ ]:
adata = sc.read(anndata_dir / "tf_ko_panel_contrastiveVI_raw_annotated.h5ad")

In [ ]:
eec_lineage = ['Early EEC Progenitors', 'Late EEC Progenitors', 'EC Cells', 'X Cells', 'D Cells', 'I/N Cells', 'K Cells']
adata_eecs = adata[adata.obs["annotation"].isin(eec_lineage)]
adata_eecs.obs['annotation'] = adata_eecs.obs['annotation'].replace({'Early EEC Progenitors': 'EEC Progenitors', 'Late EEC Progenitors': 'EEC Progenitors'})
adata_eecs.write(anndata_dir / "tf_ko_panel_contrastiveVI_raw_annotated_eec.h5ad")

In [ ]:
non_eec_lineage = ['Stem Cells', 'TA Cells', 'Cycling TA Cells', 'Enterocytes', 'Goblet Cells']
adata_non_eecs = adata[adata.obs["annotation"].isin(non_eec_lineage)]
adata_non_eecs.obs['annotation'] = adata_non_eecs.obs['annotation'].replace({'Cycling TA Cells': 'TA Cells'})
adata_non_eecs.write(anndata_dir / "tf_ko_panel_contrastiveVI_raw_annotated_non_eec.h5ad")

In [ ]:
adata = adata[adata.obs["cystic"] == "budding"]

In [ ]:
eec_lineage = ['Early EEC Progenitors', 'Late EEC Progenitors', 'EC Cells', 'X Cells', 'D Cells', 'I/N Cells', 'K Cells']
adata_eecs = adata[adata.obs["annotation"].isin(eec_lineage)]
adata_eecs.obs['annotation'] = adata_eecs.obs['annotation'].replace({'Early EEC Progenitors': 'EEC Progenitors', 'Late EEC Progenitors': 'EEC Progenitors'})
adata_eecs.write(anndata_dir / "tf_ko_panel_contrastiveVI_raw_annotated_budding_eec.h5ad")

In [ ]:
non_eec_lineage = ['Stem Cells', 'TA Cells', 'Cycling TA Cells', 'Enterocytes', 'Goblet Cells']
adata_non_eecs = adata[adata.obs["annotation"].isin(non_eec_lineage)]
adata_non_eecs.obs['annotation'] = adata_non_eecs.obs['annotation'].replace({'Cycling TA Cells': 'TA Cells'})
adata_non_eecs.write(anndata_dir / "tf_ko_panel_contrastiveVI_raw_annotated_budding_non_eec.h5ad")

### Showcase individual TFs

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import scanpy as sc

# Base colormap
base_cmap = plt.cm.GnBu

# Create a new colormap with grey for zero
colors = [(0.7, 0.7, 0.7)] + [base_cmap(i) for i in range(base_cmap.N)]
new_cmap = mcolors.LinearSegmentedColormap.from_list("grey_GnBu", colors, N=base_cmap.N + 1)

In [ ]:
sc.pl.embedding(adata, 
                basis="X_umap_shared_rep", 
                color=["LGR5", "MKI67", "APOA4", "FCGBP", "DEFA5", 
                       "NEUROG3", "GHRL", "SST", "GIP", "CCK", "TPH1"], 
                frameon=False,
                show=True,
                size=5,
                color_map=new_cmap,
                use_raw=False,
                save="_shared_rep_ind_tf_feature_exp.pdf")

In [ ]:

sc.pl.embedding(adata[adata.obs["condition"] == "Sox4"], 
                basis="X_umap_shared_rep", 
                color=["NEUROG3", "APOA4"], 
                frameon=False,
                show=True,
                use_raw=False)

In [ ]:
sc.pl.embedding(adata[adata.obs["condition"] == "Neurog3"], 
                basis="X_umap_shared_rep", 
                color=["NEUROG3", "APOA4", "FABP1"], 
                frameon=False,
                show=True,
                use_raw=False)

## Subset to budding only

In [ ]:
adata = adata[adata.obs["cystic"] == "budding"]

## Plot marker expression

In [ ]:
marker_dict = {
    "Stem Cells": ["LGR5", "SMOC2"],
    "TA Cells": ["OLFM4", "TLN2"],
    "Cycling TA Cells": ["MKI67", "TOP2A"],
    "Enterocytes": ["FABP1", "APOA4"],
    "Goblet Cells": ["FCGBP", "MUC2"],
    "Paneth Cells": ["DEFA5", "DEFA6"],
    "Early EEC Progenitors": ["NEUROG3", "ADGRL2"],
    "Late EEC Progenitors": ["RFX3",],
    "EC Cells": ["TPH1", "CHGB"],
    "D Cells": ["SST", "HHEX"],
    "X Cells": ["GHRL", "MLN"],
    "K Cells": ["GIP", "PLAGL1"],
    "I/N Cells": ["CCK", "NTS", "ONECUT3"],
}

In [ ]:
adata.obs["annotation"] = adata.obs["annotation"].astype("category").cat.reorder_categories([elem for elem in list(marker_dict.keys()) if elem in adata.obs["annotation"].unique()])

In [ ]:
adata = adata[adata.obs["annotation"].sort_values().index]

In [ ]:
sc.pl.dotplot(adata, var_names=marker_dict, groupby="annotation", standard_scale="var", cmap="Reds", figsize=(6,6), use_raw=False, show=True, swap_axes=True, save="tf_ko_panel_contrastiveVI_budding_marker_dotplot.pdf")

## Plot distribution of cell types across conditions

In [ ]:
ct_palette = {
    # proliferative axis
    "Stem Cells": "#1f77b4",          # deep blue
    "TA Cells": "#4c91c6",            # medium blue
    "Cycling TA Cells": "#7aaed6",    # light blue

    # absorptive lineage
    "Enterocytes": "#2ca02c",         # green

    # secretory lineage
    "Goblet Cells": "#ff7f0e",        # orange
    "Paneth Cells": "#ffbb78",        # light orange

    # EEC progenitors
    "Early EEC Progenitors": "#9467bd",   # purple
    "Late EEC Progenitors": "#b699d1",    # light purple

    # EEC subtypes 
    "EC Cells": "#d62728",            # strong red
    "D Cells": "#17becf",             # cyan / turquoise
    "X Cells": "#e377c2",             # magenta
    "K Cells": "#bcbd22",             # olive
    "I/N Cells": "#7f7f7f",           # dark grey
}

sc.pl.embedding(adata, basis="X_umap_shared_rep", color="annotation", palette=ct_palette, legend_loc="on data")

In [ ]:
adata.obs["batch"] = adata.obs["batch"].astype("category")

batch_palette = {
    1: "#222222",  
    2: "#555555", 
    3: "#888888",  
    4: "#BBBBBB",  
}

sc.pl.embedding(adata, basis="X_umap", color="batch", legend_loc="right", palette=batch_palette, show=False)

In [ ]:
adata.obs["condition_unique"] = adata.obs.apply(
    lambda row: f"Control_{row['sample'].replace('Cnt', '')}" if row["condition"] == "Control" else row["condition"],
    axis=1
)

In [ ]:
adata.obs["eec_split"] = adata.obs["annotation"].apply(lambda x: "EECs" if x in ["Early EEC Progenitors", "Late EEC Progenitors", "EC Cells", "D Cells", "X Cells", "K Cells", "I/N Cells", "L Cells"] else "Non-EECs")

## Plot distributions

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import marsilea as ma
import marsilea.plotter as mp

def plot_sample_cell_type_barplot(
    adata,
    groupby_sample="orig.ident",
    groupby_annotations=["day", "cafs", "donor"],
    cell_type_column="cell_type",
    normalize=True,
    figsize=(8, 3),
    palette_suffix="_colors",
    controls_grouped=False,
):
    """
    Plots a sample × cell type stacked barplot with top annotation layers from adata.obs.

    Parameters:
    -----------
    adata : AnnData
        AnnData object with `.obs` containing sample, cell type, and annotation columns.
    groupby_sample : str
        Column in `.obs` to group cells into samples (x-axis of the barplot).
    groupby_annotations : list of str
        List of columns in `.obs` to annotate above the barplot.
        Must have corresponding color palettes in `.uns` (with suffix defined below).
    cell_type_column : str
        Column in `.obs` that defines the cell types (i.e., stacked bar segments).
    normalize : bool
        Whether to normalize bar heights per sample.
    figsize : tuple
        Size of the figure.
    palette_suffix : str
        Suffix used in `.uns` to find color palettes, e.g., 'day' → 'day_colors'.
    controls_grouped : bool
        Whether to group control samples together at the start of the plot.
    """
    
    # Cell type count matrix: sample x cell_type
    data = (
        adata.obs[[groupby_sample, cell_type_column]]
        .groupby([groupby_sample, cell_type_column])
        .size()
        .unstack(fill_value=0)
        .T
    )
    
    # Metadata: one row per sample
    sample_metadata = adata.obs.groupby(groupby_sample)[groupby_annotations].first()

    # Stacked bar (normalized per sample if required)
    bar_data = data / data.sum(axis=0) if normalize else data
    bar = mp.StackBar(bar_data, width=0.9, colors=adata.uns[f"{cell_type_column}{palette_suffix}"], legend_kws=dict(title="Cell Type"))

    # Top annotation layers
    top_layers = []
    for col in groupby_annotations:
        if col not in sample_metadata.columns:
            continue
        palette_key = f"{col}{palette_suffix}"
        if palette_key not in adata.uns:
            raise ValueError(f"Missing palette in `.uns`: '{palette_key}'")
        categories = adata.obs[col].cat.categories
        palette = dict(zip(categories, adata.uns[palette_key]))
        top_layers.append(mp.Colors(sample_metadata[col], palette=palette, label=col.capitalize()))

    # Plotting
    wb = ma.ClusterBoard(cluster_data=data, width=figsize[0], height=figsize[1])
    wb.add_layer(bar)
    for layer in top_layers:
        wb.add_top(layer, size=0.2, pad=0.2)
    wb.add_dendrogram("top")
    wb.add_bottom(mp.Labels(data.columns, rotation=90, fontsize=8))
    
    if controls_grouped:
        groups = [1 if "Control" in x else 0 for x in data.columns]
        wb.group_cols(groups)
    wb.add_legends()
        
    with plt.rc_context(rc={"axes.grid": False, "grid.color": ".8"}):
        wb.render()
    return wb

#### Non-EECs

In [ ]:
adata_non_eecs = adata[adata.obs["eec_split"] != "EECs"]

In [ ]:
plot1 = plot_sample_cell_type_barplot(adata_non_eecs[adata_non_eecs.obs["condition"] == "Control"], 
                              groupby_sample="condition_unique",
                              groupby_annotations=["batch"],
                              cell_type_column="annotation",
                              figsize=(2, 3),
                              )


with plt.rc_context(rc={"axes.grid": False, "grid.color": ".8"}):

    main_ax = plot1.get_main_ax()
    main_ax.set_autoscale_on(False)
    main_ax.set_ylim(0, 1)

    plt.savefig(
        Path(sc.settings.figdir) / "tf_ko_panel_contrastiveVI_non_eecs_control_samplecelltype_barplot.pdf",
        bbox_inches="tight"
    )


In [ ]:
plot2 = plot_sample_cell_type_barplot(adata_non_eecs[adata_non_eecs.obs["condition"] != "Control"], 
                              groupby_sample="condition_unique",
                              groupby_annotations=["batch"],
                              cell_type_column="annotation",
                              figsize=(12, 3),
                              )

with plt.rc_context(rc={"axes.grid": False, "grid.color": ".8"}):
    
    main_ax = plot2.get_main_ax()
    main_ax.set_ylim(0, 1)

    plt.savefig(Path(sc.settings.figdir) / "tf_ko_panel_contrastiveVI_non_eecs_perturbed_cond_sample_celltype_barplot.pdf")

In [ ]:
plot3 = plot_sample_cell_type_barplot(adata_non_eecs, 
                              groupby_sample="condition_unique",
                              groupby_annotations=["batch"],
                              cell_type_column="annotation",
                              figsize=(12, 3),
                              controls_grouped=True
                              )

with plt.rc_context(rc={"axes.grid": False, "grid.color": ".8"}):

    plt.savefig(
        Path(sc.settings.figdir) / "tf_ko_panel_contrastiveVI_non_eecs_all_sample_celltype_barplot.pdf",
        bbox_inches="tight"
    )

#### EECs

In [ ]:
adata_eecs = adata[adata.obs["eec_split"] == "EECs"]

In [ ]:
counts = (
    pd.crosstab(adata_eecs.obs["condition"], adata_eecs.obs["annotation"])
    .sum(axis=1)
    .sort_values(ascending=True)
)

counts.plot(kind="barh", figsize=(4,8), color="steelblue", edgecolor="black")

plt.xlabel("Total cells")
plt.ylabel("Condition")
plt.title("Total cells per condition")

plt.grid(False)  # remove grid
plt.tight_layout()
plt.show()

In [ ]:
# Remove conditions with very few cells i.e. Atoh1 & Neurog3
adata_eecs = adata_eecs[adata_eecs.obs["condition"].isin(counts[counts > 50].index)]

In [ ]:
plot1 = plot_sample_cell_type_barplot(adata_eecs[adata_eecs.obs["condition"] == "Control"], 
                              groupby_sample="condition_unique",
                              groupby_annotations=["batch"],
                              cell_type_column="annotation",
                              figsize=(5, 3),
                              )
with plt.rc_context(rc={"axes.grid": False, "grid.color": ".8"}):
    plot1.save(Path(sc.settings.figdir) / "tf_ko_panel_contrastiveVI_eecs_control_sample_celltype_barplot.pdf")

In [ ]:
plot2 = plot_sample_cell_type_barplot(adata_eecs[adata_eecs.obs["condition"] != "Control"], 
                              groupby_sample="condition_unique",
                              groupby_annotations=["batch"],
                              cell_type_column="annotation",
                              figsize=(12, 3),
                              )
with plt.rc_context(rc={"axes.grid": False, "grid.color": ".8"}):
    plot2.save(Path(sc.settings.figdir) / "tf_ko_panel_contrastiveVI_eecs_perturbed_cond_sample_celltype_barplot.pdf")

In [ ]:
plot3 = plot_sample_cell_type_barplot(adata_eecs, 
                              groupby_sample="condition_unique",
                              groupby_annotations=["batch"],
                              cell_type_column="annotation",
                              figsize=(12, 3),
                              controls_grouped=True
                              )

with plt.rc_context(rc={"axes.grid": False, "grid.color": ".8"}):

    plt.savefig(
        Path(sc.settings.figdir) / "tf_ko_panel_contrastiveVI_eecs_all_sample_celltype_barplot.pdf",
        bbox_inches="tight"
    )

In [ ]:
number_of_cells_per_condition_and_annotation = pd.crosstab(adata.obs["condition_unique"], adata.obs["annotation"], margins=True, margins_name="Total")
pct_of_cells_per_condition_and_annotation = pd.crosstab(adata.obs["condition_unique"], adata.obs["annotation"], margins=True, margins_name="Total", normalize=True)

number_of_cells_per_condition_and_annotation.to_csv(Path(settings.ANALYSIS.tables_dir) / "tf_ko_panael_contrastiveVI_number_of_cells_per_condition_and_annotation.csv")
pct_of_cells_per_condition_and_annotation.to_csv(Path(settings.ANALYSIS.tables_dir) / "tf_ko_panel_contrastiveVI_pct_of_cells_per_condition_and_annotation.csv")